# Lesson 18: Low-Code AutoML & Interactive Gradio Deployment (Heart Failure Prediction)

In this notebook, you will:
1. Load a self-contained 299-row **Heart Failure Clinical Records** dataset (12 features + `DEATH_EVENT`).
2. Run open-source **FLAML AutoML** optimizing `roc_auc` on Google Colab Free Tier.
3. Launch an interactive **Gradio** clinical prediction web app.

In [ ]:
!pip install -q flaml scikit-learn pandas numpy gradio
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from flaml import AutoML

# Self-contained 299-patient Heart Failure Clinical Records generator matching dataset distributions
rng = np.random.default_rng(18)
n = 299
age = rng.integers(40, 95, size=n)
ef = rng.integers(14, 80, size=n)
sc = np.round(rng.lognormal(0.2, 0.5, size=n), 2)
time_days = rng.integers(4, 285, size=n)
logit = 0.04 * (age - 60) - 0.05 * (ef - 38) + 0.7 * (sc - 1.3) - 0.02 * (time_days - 130)
death_event = (rng.random(n) < (1 / (1 + np.exp(-logit)))).astype(int)

df = pd.DataFrame({
    "age": age, "anaemia": rng.integers(0, 2, n), "creatinine_phosphokinase": rng.integers(50, 2500, n),
    "diabetes": rng.integers(0, 2, n), "ejection_fraction": ef, "high_blood_pressure": rng.integers(0, 2, n),
    "platelets": rng.integers(100000, 450000, n), "serum_creatinine": sc, "serum_sodium": rng.integers(125, 145, n),
    "sex": rng.integers(0, 2, n), "smoking": rng.integers(0, 2, n), "time": time_days,
    "DEATH_EVENT": death_event
})

X = df.drop(columns=["DEATH_EVENT"])
y = df["DEATH_EVENT"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

automl = AutoML()
automl.fit(X_train, y_train, task="classification", metric="roc_auc", time_budget=15, verbose=0)
print("Best AutoML estimator:", automl.best_estimator)
print("Test ROC-AUC:", round(1 - automl.best_loss, 3))

In [ ]:
import gradio as gr

def predict_risk(age, ejection_fraction, serum_creatinine, time):
    row = pd.DataFrame([{
        "age": age, "anaemia": 0, "creatinine_phosphokinase": 500, "diabetes": 0,
        "ejection_fraction": ejection_fraction, "high_blood_pressure": 0, "platelets": 260000,
        "serum_creatinine": serum_creatinine, "serum_sodium": 137, "sex": 0, "smoking": 0, "time": time
    }])
    prob = float(automl.predict_proba(row)[0][1])
    return f"Predicted Risk: {'HIGH (True)' if prob >= 0.5 else 'LOW (False)'} (p={prob:.2%})"

demo = gr.Interface(
    fn=predict_risk,
    inputs=[gr.Number(value=60, label="Age"), gr.Number(value=38, label="Ejection Fraction"), gr.Number(value=1.4, label="Serum Creatinine"), gr.Number(value=130, label="Follow-up Days")],
    outputs="text",
    title="Heart Failure Risk Predictor (Open-Source AutoML)"
)
# demo.launch(share=True)  # Uncomment in Google Colab for a live public URL
print("Sample check:", predict_risk(60, 38, 1.4, 130))